# Reproduction of Tutorial 1: Active inference from scratch

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/infer-actively/pymdp/blob/main/docs/notebooks/active_inference_from_scratch.ipynb)

*Author: Theo Pana*

This is a reproduction of Conor Heins's active inference tutorial.

In [ ]:
# pip install inferactively-pymdp

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Define some auxiliary functions
for plotting

In [ ]:
def plot_likelihood(matrix, xlabels = list(range(9)), ylabels = list(range(9)), title_str = "Likelihood distribution (A)"):
    """
    Plots a 2-D likelihood matrix as a heatmap
    """
    
    if not np.isclose(matrix.sum(axis=0), 1.0).all():
        raise ValueError("Distribution not column-normalized! Please normalize (ensure matrix.sum(axis=0) == 1.0 for all columns!)")
    
    plt.figure(figsize = (6,6))
    sns.heatmap(matrix, xticklabels = xlabels, yticklabels = ylabels, cmap = 'gray', cbar = False, vmin = 0.0, vmax = 1.0)
    plt.title(title_str)
    plt.show()
    
def plot_grid(grid_locations, num_x = 3, num_y = 3):
    """
    Plots the spatial coordinates of GridWorld as a heatmap, with each (X, Y) coordinate
    labeled with its linear index (its `state id`)
    """
    
    grid_heatmap = np.zeros((num_x, num_y))
    for linear_idx, location in enumerate(grid_locations):
        y, x = location
        grid_heatmap[y, x] = linear_idx
    sns.set(font_scale=1.5)
    sns.heatmap(grid_heatmap, annot=True, cbar = False, fmt='.0f', cmap='crest')
        
def plot_point_on_grid(state_vector, grid_locations):
    """
    Plots the current location of the agent on the grid world
    """
    state_index = np.where(state_vector)[0][0]
    y, x = grid_locations[state_index]
    grid_heatmap = np.zeros((3,3))
    grid_heatmap[y,x] = 1.0
    sns.heatmap(grid_heatmap, cbar = False, fmt='.0f')
    
def plot_beliefs(belief_dist, title_str=""):
    """
    Plot a categorical distribution or belief distribution, stored in the 1-D numpy vector `belief_dist`
    """
    
    if not np.isclose(belief_dist.sum(), 1.0):
        raise ValueError("Distribution not normalized! Please normalize")
    
    plt.grid(zorder=0)
    plt.bar(range(belief_dist.shape[0]), belief_dist, color='r', zorder=3)
    plt.xticks(range(belief_dist.shape[0]))
    plt.title(title_str)
    plt.show()
    

# Categorical distributions

In [ ]:
from pymdp import utils

In [ ]:
my_categorical = np.random.rand(3)
# my_categorical = np.array([0.5, 0.3, 0.8]) # can write other numbers
my_categorical = utils.norm_dist(my_categorical) # normalize the distribution so it integrates to 1.0

print(my_categorical.reshape(-1, 1)) # we reshape it to display as a column vector
print(f'Integral of the distribution: {round(my_categorical.sum(), 2)}')

In [ ]:
# We can sample a discrete outcome from this distribution with `sample()` from the `utils` module

sampled_outcome = utils.sample(my_categorical)
print(f'Sampled outcome: {sampled_outcome}')

In [ ]:
# Now plot the beliefs using `plot_beliefs()`, defined in the beginning

plot_beliefs(my_categorical, title_str = "A random (unconditional) categorical distribution")

In [ ]:
# Now moving on to *conditional* categorical distributions or likelihoods 
# -- i.e. how we represent the distribution of one discrete random variable X,
# conditioned on the settings of another discrete random variable Y.
# Mathematically: P(X|Y)

# Initialize it with random numbers
p_x_given_y = np.random.rand(3, 4)
print(p_x_given_y.round(3))

In [ ]:
# Normalize it
p_x_given_y = utils.norm_dist(p_x_given_y)
print(p_x_given_y)

In [ ]:
# Print the first column (i.e. P(X|Y) = 0) and show it's a proper categorical distribution
print(p_x_given_y[:,0].reshape(-1,1))
print(f'Integral of P(X|Y=0): {p_x_given_y[:,0].sum()}')

In [ ]:
# So column `i` of the matrix `p_x_given_y` represents the conditional probability of X, 
# given the i-th level of the random variable Y -- i.e. P(X|Y = i)

# Next: taking expectations of random variables using matrix-vector products

""" Create a P(Y) and P(X|Y) using the same numbers from the slides """

p_y = np.array([0.75, 0.25]) # already normalized no need to `utils.norm_dist()` it

# columns also already normalized
p_x_given_y = np.array([[0.6, 0.5],
                        [0.15, 0.41],
                        [0.25, 0.09]])

print(p_y.round(3).reshape(-1,1))
print(p_x_given_y.round(3))

In [ ]:
# Calculate expected value of X, given our belief about Y, P(Y) using a simple vector-matric product, of the form Ax

# first method of dot product: numpy array
E_x_wrt_y = p_x_given_y.dot(p_y)

# second method of dot product: function np.dot with two arguments
# E_x_wrt_y = np.dot(p_x_given_y, p_y)

In [ ]:
# Now print and verify the result is a proper categorical distribution (integrates to 1.0)

print(E_x_wrt_y)
print(f'Integral: {E_x_wrt_y.sum().round(3)}')

### Grid-world environment

In [ ]:
import itertools

""" Create the grid locations in the form of (Y, X) tuples """
grid_locations = list(itertools.product(range(3), repeat = 2))
print(grid_locations)

In [ ]:
plot_grid(grid_locations)

basically, the likelihood matrix A is (O, S) where each O is an observation and each S is a state. It is probability of seeing observation O, given you're in state S: A[o, s] = P(o | s).

A transition matrix B is (S', S) that maps how one action u may transition from current state S -> future state S'. So if the action UP has 9 possible future states S' and 9 possible current states S, B is a 9x9 transition matrix: B[s', s] = P(s' | s) for action u. 
If there are four actions U (eg. up, down, left, right), B is a 9x9x4 transition tensor: B[s', s, u] = P(s' | s, u)

# Building the generative model: A, B, C, and D

## 1. The A matrix, P(o | s)
The generative model's "prior beliefs" about how hidden states relate to observations

In [ ]:
""" Create variables for storing the dimensionalities of the hidden states and observations """

n_states = len(grid_locations)
n_observations = len(grid_locations)

print(f'Dimensionality of hidden states: {n_states}')
print(f'Dimensionality of observations: {n_observations}')

In [ ]:
""" Create the A matrix """

A = np.zeros( (n_states, n_observations))

In [ ]:
""" Create an unambiguous or 'noise-less' mapping between hidden states and observations """

np.fill_diagonal(A, 1.0)

# alternative:
# A = np.eye(n_observations, n_states)

Likelihood matrices show relationships between indices of the state space, which are indexed linearly. 
Remember the (Y, X) coordinates of the grid world don't necessarily map neatly onto the locations in 
the matrix here, which has "unwrapped" the Grid World's (Y, X) locations into columns/rows


In [ ]:
plot_likelihood(A, title_str = "A matrix or $P(o|s)$") 

How could we introduce uncertainty into this matrix? How do we introduce ambiguity or uncertainty into the agent's model of the world?

In [ ]:
""" Remind yourself of the mapping between linear indices (0 through 8) and grid locations (Y, X) """
plot_grid(grid_locations)

In [ ]:
A_noisy = A.copy()

# Hard code some probabilities.

# The probability of seeing yourself at location 0, given you're in location 0, is 1/3 -- P(o == 0 | s == 0) = 0.333...
A_noisy[0,0] = 1/3.0 # Location (0,0)

# The probability of seeing yourself in location 1, given you're in location 0, is 1/3 -- P(o == 1 | s == 0) = 0.333...
A_noisy[1,0] = 1/3.0 # Location (1,0): one step right from state 0

A_noisy[3,0] = 1/3.0 # Location (3,0): one step down from state 0

In [ ]:
plot_likelihood(A_noisy, title_str='modified A matrix where location (0,0) is "blurry"')

In [ ]:
""" Let's make one grid location "ambiguous": ie. it could be easily confused with neighbouring locations """
my_A_noisy = A_noisy.copy()

# locations 3 and 7 are nearest neighbours to location 6
my_A_noisy[3,6] = 1.0 / 3.0
my_A_noisy[6,6] = 1.0 / 3.0
my_A_noisy[7,6] = 1.0 / 3.0

# (make sure the probabilities sum to 1 for each column)

In [ ]:
# Plot the new A matrix, which now has two "blurry" locations
plot_likelihood(my_A_noisy, title_str = "Noisy A matrix now with TWO ambiguous locations")

## The B matrix: P(s' | s, u) or $P(s_t \mid s_{t-1}, u_{t-1})$
The generative model's "prior beliefs" about (controllable) transitions between hidden states over time. Namely, how do hidden states at time $t$ result from hidden states at some previous time $t - 1$. These transition dynamics are conditioned on some past action $u_{t-1}$

In [ ]:
actions = ["UP", "DOWN", "LEFT", "RIGHT", "STAY"] # 5 actions

def create_B_matrix():
    B = np.zeros( (len(grid_locations), len(grid_locations), len(actions)) )
    
    for action_id, action_label in enumerate(actions):
        
        for curr_state, grid_location in enumerate(grid_locations):
            
            y, x = grid_location
            
            if action_label == "UP":
                next_y = y - 1 if y > 0 else y
                next_x = x
            elif action_label == "DOWN":
                next_y = y + 1 if y < 2 else y
                next_x = x
            elif action_label == "LEFT":
                next_x = x - 1 if x > 0 else x
                next_y = y
            elif action_label == "RIGHT":
                next_x = x + 1 if x < 2 else x
                next_y = y
            elif action_label == "STAY":
                next_x = x
                next_y = y
            new_location = (next_y, next_x)
            next_state = grid_locations.index(new_location)
            B[next_state, curr_state, action_id] = 1.0
    return B

B = create_B_matrix()

In [ ]:
# Now "take" an action, using the matrix-vector product of an action-conditioned "slice" of the B array and a previous state vector

""" Define a starting location """
starting_location = (1,0)

""" get the linear index of the state """
state_index = grid_locations.index(starting_location)

""" and create a state vector out of it """
starting_state = utils.onehot(state_index, n_states)

plot_point_on_grid(starting_state, grid_locations)

In [ ]:
plot_beliefs(starting_state, "Categorical distribution over the starting state")

In [ ]:
# Now imagine moving RIGHT - write the conditional expectation, that will create the state vector corresponding to the new state after taking a step to the right.

""" Redefine the actions here, just for reference """
actions = ["UP", "DOWN", "LEFT", "RIGHT", "STAY"]

""" Generate the next state vector, given the starting state and the B transition matrix """
right_action_idx = actions.index("RIGHT")
next_state = B[:,:, right_action_idx].dot(starting_state) # input the indices to the B matrix

""" Plot the next state, after taking the action"""
plot_point_on_grid(next_state, grid_locations)

In [ ]:
# Now imagine moving DOWN from where we just landed - write the conditional expectation, that will create the state vector corresponding to the new state after stepping down.

""" Generate the next state vector, given the previous state and the B matrix """
prev_state = next_state.copy()
down_action_index = actions.index("DOWN")
next_state = B[:,:, down_action_index].dot(prev_state)

""" Plot the new state vector, after making the DOWN movement """
plot_point_on_grid(next_state, grid_locations)

### 3. The prior over observations: the C vector or $\tilde{P}(o)$

The (biased) generative model's prior preference for particular observations, encoded in terms of probabilities.

In [ ]:
""" Create an empty vector to store the preferences over observations """
C = np.zeros(n_observations)

""" Choose an observation index to be the 'desired' rewarding index, and fill out the C vector accordingly """
desired_location = (2,2) # Choose a desired location
desired_location_index = grid_locations.index(desired_location) # get the linear index of the grid location; in terms of 0 through 8

C[desired_location_index] = 1.0 # set the preference for that location to be 100%

""" Let's look at the prior preference distribution """
plot_beliefs(C, title_str = "Preferences over observations")

### 4. The prior over hidden states: the D vector or P(s)

The generative model's prior belief over hidden states at the first timestep

In [ ]:
""" Create a D vector, basically a belief that the agent has about its own starting location """

# create a one-hot (certain) belief about initial state
D = utils.onehot(0, n_states)

# demonstrate how belief about initial state can also be uncertain / spread among different possible initial states
# alternative, where you have a degenerate/noisy prior belief
# D = utils.norm_dist(np.ones(n_states))

""" Let's look at the prior over hidden states """
plot_beliefs(D, title_str = "Prior beliefs over states")

## Hidden state inference

Hidden state inference finds the setting of the optimal variational posterior $q(s_t)$ that minimizes the variational free energy. For the simple POMDP (Partially Observable Markov Decision Process) generative model and posterior we're considering here, this update reduces to a remarkably simple update rule, that is essentially proportional to Bayes' rule. 

POMDP is the generative model the agent uses to describe its world. It includes the following:
- Hidden states $s_t$: The actual state of the world -- here, the agent's actual location. The agent doesn't know this.
- Obserations $o_t$: What the agent sees. The A matrix $P(o_t\mid s_t)$ maps observation -> hidden state.
- Actions / control states $u_t$: What the agent does. The B matrix $P(s_{t+1}\mid s_t, u_t)$ maps action x (hidden_state_1 -> hidden_state_2)
- Markov: The next state depends only on current state and action; nothing earlier.
- Partially observable: since observations can be noisy/ambiguous, the agent must infer $s_t$. The agent does this with belief $q(s_t)$

Actie inference also adds:
- C: Preferences over observations, which replaces rewards.
- D: The prior over the initial state (agent's believed starting location)

The goal is for the agent to guess where it is: $s_t$ from an observation $o_t$. It does this by keeping a belief, holding its location probability for each cell, $q(s_t)$. Each timestep, the agent updates its believed location.

Varational free energy (VFE) scores how good a belief is (low FE good). A belief has scores low FE based on two things:
- How well it explains the current observation (ie. fits the A matrix; average log-likelihood of the observation under the belief),
- and how well it aligns with the prior $p$ (proximity to agent's expectation before looking; measured with KL divergence (Kullback-Leibler divergence) between probability distributions).
Hidden state inference is done by picking the belief $q(s_t)$ with the lowest FE. "Variational" means searching over candidate beliefs $q$. 

Thus, KL divergence measures how far candidate belief $q$ starys from prior belief $p$:
$$\text{KL}\big(q ,|, p\big) = \sum_s q(s),\ln\frac{q(s)}{p(s)}$$

The full FE score formula sums two terms. KL divergence and a second one. The second term is how well $q$ "fits" the A matrix, measured by the average log-likelihood of the observation under the belief. It's subtracted; explaining the observation well lowers $F$. The two added terms pull in opposite directions, and the belief that balances them best is Bayes' rule's answer. 

---

Example:
prior = `[0.10, 0.70, 0.20]`

$A$ row = `[0.90, 0.10, 0.10]`

Here's three candidate beliefs:

$q_1$ = `[0.10, 0.70, 0.20]` # Stick with priors $\mid$ KL (stray) = 0.00 $\mid$ Fit = -2.08 $\mid$ $F$ = 2.08

$q_2$ = `[1.00, 0.00, 0.00]` # Go all-in on cell 1 $\mid$ KL (stray) = 2.30 $\mid$ Fit = -0.11 $\mid$ $F$ = 2.41

$q_3$ = `[0.50, 0.39, 0.11]` # Bayes answer $\mid$ KL (stray) = 0.51 $\mid$ Fit = -1.20 $\mid$ $F$ = 1.71

Bayes answer $q_3$ wins


---

Q-learning RL, by contrast, usually assumes a fully observable MDP, where the agent knows $s_t$ and therefore needs no inference step.

So. **Hidden State Inference** finds the setting of the optimal variational posterior $q(s_t)$ that minimizes VFE. For the simple POMDP gen model and posterior we're using here, the update rule is simple, and essentially proportional to Bayes' rule:

$q(s_t) = \sigma(ln A[o,:] + ln B[:,:,u] \cdot q(s_{t-1}))$

$B[:,:,u] \cdot q(s_{t-1})$ = the prior (predict)
- $B$ is 3D: B[next_state, current_state, action]
- $B[:,:,u]$ picks the transition matrix for the action $u$ the agent just took. Column $j$ says where you end up if you start in state $j$.

$A[o,:]$ = the likelihood (observe)
- $A$ is $A[observation, state]$
- With the fixed observation $o$, $A[o,:]$ is a row: for each state, how likely is it to see $o$ from there.

$ln(...) + ln(...)$ = multiply, in log space. Same as multiplying likelihood * prior.

$\sigma$ = softmax, which undoes the log and normalizes.

In [ ]:
# Import softmax and find a "numerically-stable" version of np.log from pymdp.maths

from pymdp.maths import softmax
from pymdp.maths import spm_log_single as log_stable

In [ ]:
# Implement infer_states() function:

""" Create an infer states function that implements the math we just discussed """

def infer_states(observation_index, A, prior):
    
    """ Implement function here -- NOTE: prior is already passed in, so no need to do anything with B matrix. """
    """ This function has already been given P(s_t). The conditional expectation that creates "today's prior", using "yesterday's posterior", will 
        happen *before calling* this function. """
        
    log_likelihood = log_stable(A[observation_index,:])
    
    log_prior = log_stable(prior)
    
    qs = softmax(log_likelihood + log_prior)
    
    return qs

In [ ]:
# Let's do a hidden simulated timestep of hidden state inference

qs_past = utils.onehot(4, n_states) # Agent beliefs they were at location 4 -- (1,1) -- one timestep ago

last_action = "UP" # agent knows it moved UP one timestep ago
action_id = actions.index(last_action)

Now, get "today's prior" using "yesterday's posterior" and the past action, ie:

$P(s_t) = E_{q(s_{t-1})}[P(s_t \mid s_{t-1},u_{t-1})]$

and choose an observation that is consistent with the new location

In [ ]:
prior = B[:,:,action_id].dot(qs_past)

observation_index = 1

In [ ]:
# Now run infer_states() using the observation, A matrix, and the prior we just calculated above

qs_new = infer_states(observation_index, A, prior)
plot_beliefs(qs_new, title_str = "Beliefs about hidden states")

In [ ]:
# Now let's see what happens if observation and prior disagree with eachother?

""" Get an observation that 'conflicts' with the prior """
observation_index = 2 # this is like the agent seeing itself in location (0, 2)
qs_new = infer_states(observation_index, A, prior)
plot_beliefs(qs_new)

In [ ]:
# Now what happens if likelihood and prior are partially ambiguous?

""" Create an ambiguous A matrix """
A_partially_ambiguous = softmax(A)
print(A_partially_ambiguous.round(3))

In [ ]:
""" ... and a noisy prior """
noisy_prior = softmax(prior)
plot_beliefs(noisy_prior)